<a href="https://colab.research.google.com/github/wunann03/AmplifAi-Bootcamp-Basics-of-Neural-Networks/blob/Forward-Propagation-Exercise/Copy_of_Classification_Task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torch

In [2]:
!curl -L -o heart_disease.zip\
 https://www.kaggle.com/api/v1/datasets/download/fedesoriano/heart-failure-prediction
!unzip  heart_disease.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  8762  100  8762    0     0  12411      0 --:--:-- --:--:-- --:--:-- 75534
Archive:  heart_disease.zip
  inflating: heart.csv               


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

In [4]:
# --- 1. Define the Neural Network Model ---
class SimpleNN(nn.Module):
    def __init__(self, input_size, hidden_size1, hidden_size2, output_size):
        super(SimpleNN, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_size1),
            nn.ReLU(),
            nn.Linear(hidden_size1, hidden_size2),
            nn.ReLU(),
            nn.Linear(hidden_size2, output_size),
            nn.Sigmoid()  # Use Sigmoid for binary classification output (0-1)
        )

    def forward(self, x):
        return self.network(x)

In [5]:
# --- 2. Load and Prepare the Data ---
try:
    df = pd.read_csv('heart.csv')
    print("Dataset loaded successfully from 'heart_disease.csv'.")
except FileNotFoundError:
    print("Error: 'heart_disease.csv' not found.")
    print("Please make sure the file is in the same directory as this script.")
    exit()

# Separate features (X) and target (y)
target = 'HeartDisease'
X = df.drop(columns=[target])
y = df[target].values.reshape(-1, 1)

# Identify numerical and categorical features
numerical_features = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']
categorical_features = ['Sex', 'ChestPainType', 'FastingBS', 'RestingECG', 'ExerciseAngina', 'ST_Slope']

# Handle categorical features using one-hot encoding
X_encoded = pd.get_dummies(X, columns=categorical_features, drop_first=True)

# Split the data into training and testing sets
X_train_df, X_test_df, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

# Normalize numerical features using StandardScaler
scaler = StandardScaler()
X_train_df[numerical_features] = scaler.fit_transform(X_train_df[numerical_features])
X_test_df[numerical_features] = scaler.transform(X_test_df[numerical_features])

# Convert NumPy arrays to PyTorch tensors. We explicitly cast the numpy arrays to a float type
# to prevent the "object_" type error.
X_train_tensor = torch.tensor(X_train_df.values.astype(np.float32), dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_df.values.astype(np.float32), dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)


Dataset loaded successfully from 'heart_disease.csv'.


In [6]:
# --- 3. Instantiate Model, Loss Function, and Optimizer ---
input_size = X_train_tensor.shape[1] # Number of features after one-hot encoding
hidden_size1 = 128
hidden_size2 = 64
output_size = 1

model = SimpleNN(input_size, hidden_size1, hidden_size2, output_size)
criterion = nn.BCELoss()  # Binary Cross-Entropy Loss for binary classification
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [7]:
# --- 4. Train the Neural Network ---
num_epochs = 5000
print("\nTraining the model...")
for epoch in range(num_epochs):
    model.train()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 500 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

print("Training complete!")


Training the model...
Epoch [500/5000], Loss: 0.0049
Epoch [1000/5000], Loss: 0.0006
Epoch [1500/5000], Loss: 0.0002
Epoch [2000/5000], Loss: 0.0001
Epoch [2500/5000], Loss: 0.0001
Epoch [3000/5000], Loss: 0.0000
Epoch [3500/5000], Loss: 0.0000
Epoch [4000/5000], Loss: 0.0000
Epoch [4500/5000], Loss: 0.0000
Epoch [5000/5000], Loss: 0.0000
Training complete!


In [8]:
# --- 5. Evaluate the Model on the Test Set ---
model.eval()
with torch.no_grad():
    test_outputs = model(X_test_tensor)
    test_loss = criterion(test_outputs, y_test_tensor)

    # Convert probabilities to predicted classes (0 or 1)
    predicted_classes = (test_outputs > 0.5).float()

    # Calculate accuracy
    accuracy = accuracy_score(y_test_tensor, predicted_classes)

    print("\n--- Model Evaluation ---")
    print(f'Test Loss (Binary Cross-Entropy): {test_loss.item():.4f}')
    print(f'Accuracy: {accuracy:.4f}')
    print("\nClassification Report:")
    print(classification_report(y_test_tensor, predicted_classes, target_names=['No Heart Disease', 'Heart Disease']))



--- Model Evaluation ---
Test Loss (Binary Cross-Entropy): 2.3193
Accuracy: 0.8533

Classification Report:
                  precision    recall  f1-score   support

No Heart Disease       0.80      0.87      0.83        77
   Heart Disease       0.90      0.84      0.87       107

        accuracy                           0.85       184
       macro avg       0.85      0.86      0.85       184
    weighted avg       0.86      0.85      0.85       184



In [9]:
# --- 6. Make a Prediction for a new patient ---
new_patient_data = {
    'Age': 55,
    'Sex': 'M',
    'ChestPainType': 'ATA',
    'RestingBP': 130,
    'Cholesterol': 240,
    'FastingBS': 0,
    'RestingECG': 'Normal',
    'MaxHR': 150,
    'ExerciseAngina': 'N',
    'Oldpeak': 1.0,
    'ST_Slope': 'Up'
}
print("\nPredicting for a new patient with the following features:")
for key, value in new_patient_data.items():
    print(f"{key}: {value}")

# Create a DataFrame for the new data point
new_df = pd.DataFrame([new_patient_data])

# Apply the same one-hot encoding to the new data, ensuring all columns are present
new_df_encoded = pd.get_dummies(new_df, columns=categorical_features, drop_first=True)
# Align columns to ensure the same structure as training data
missing_cols = set(X_train_df.columns) - set(new_df_encoded.columns)
for c in missing_cols:
    new_df_encoded[c] = 0
new_df_encoded = new_df_encoded[X_train_df.columns]

# Normalize numerical features using the *same* scaler
new_df_encoded[numerical_features] = scaler.transform(new_df_encoded[numerical_features])

# Convert to tensor and make prediction
new_tensor = torch.tensor(new_df_encoded.values, dtype=torch.float32)
model.eval()
with torch.no_grad():
    prediction_prob = model(new_tensor).item()

predicted_class = 1 if prediction_prob > 0.5 else 0
predicted_label = 'Heart Disease' if predicted_class == 1 else 'No Heart Disease'

print(f"\nPredicted probability of heart disease: {prediction_prob:.4f}")
print(f"Prediction: {predicted_label}")


Predicting for a new patient with the following features:
Age: 55
Sex: M
ChestPainType: ATA
RestingBP: 130
Cholesterol: 240
FastingBS: 0
RestingECG: Normal
MaxHR: 150
ExerciseAngina: N
Oldpeak: 1.0
ST_Slope: Up

Predicted probability of heart disease: 0.0002
Prediction: No Heart Disease
